# Weeks 1–2 · Advanced Python for Data Science

**Programming for Data Science (606315)** · University of Petra

You already know Python. This notebook is about writing it the way data scientists do:
compact, lazy where it matters, robust when the data is dirty, and readable six months later.

| Section | Topic | Why it matters for data work |
|---|---|---|
| 1 | Comprehensions | Transform collections in one readable line |
| 2 | Iterators & generators | Process files bigger than memory |
| 3 | `lambda`, `map`, `filter`, `sorted` | Functional one-liners you will see in every Pandas codebase |
| 4 | Decorators | Timing, caching and logging without touching the function body |
| 5 | Error handling | Dirty data *will* break your code; decide what happens then |
| 6 | Type hints | Self-documenting functions that your editor can check |
| 7 | Context managers | Files, connections and timers that always clean up |

**How to use this notebook in VS Code:** click a code cell and press `Shift+Enter` to run it.
Change things. Break things. The *Variables* panel (top toolbar) shows every object in memory.

In [ ]:
import sys, platform
print(sys.version)
print("Running on:", platform.node())

## 1 · Comprehensions

A comprehension builds a collection from an iterable in a single expression.
Compare the loop with the comprehension: same result, half the lines, and the *intent* is visible immediately.

In [ ]:
gpas = [3.4, 2.1, 3.9, 2.8, 1.9, 3.2]

# the loop way
scaled = []
for g in gpas:
    scaled.append(round(g / 4 * 100, 1))

# the comprehension way
scaled2 = [round(g / 4 * 100, 1) for g in gpas]

print(scaled == scaled2, scaled2)

### Filtering with `if`, transforming with `if/else`

The `if` **after** the `for` filters elements out; an `if/else` **before** the `for` chooses a value for every element.

In [ ]:
honours = [g for g in gpas if g >= 3.0]                        # filter
status  = ["pass" if g >= 2.0 else "probation" for g in gpas]  # map with a condition
print(honours)
print(status)

### Dictionary and set comprehensions

The same syntax with braces gives you `dict` and `set`. These are the workhorses for building lookup tables.

In [ ]:
students = ["Ahmad", "Sara", "Omar", "Lina"]
gpa_by_name = {name: g for name, g in zip(students, gpas)}
majors = ["DS", "CS", "DS", "AI", "CS"]
unique_majors = {m for m in majors}

print(gpa_by_name)
print(unique_majors)
# inverting a dictionary — a classic data-cleaning move
code_to_name = {"DS": "Data Science", "CS": "Computer Science"}
name_to_code = {v: k for k, v in code_to_name.items()}
print(name_to_code)

### Nested comprehensions — use sparingly

Flattening a list of lists is the one nested comprehension worth memorising.
Read it in the same order you would write the loops.

In [ ]:
grades_per_course = [[88, 92], [75], [60, 70, 65]]
flat = [g for course in grades_per_course for g in course]
print(flat)

# A 3×3 identity matrix without NumPy (we get NumPy next week)
identity = [[1 if r == c else 0 for c in range(3)] for r in range(3)]
identity

> **Rule of thumb.** If a comprehension does not fit on one line, or needs a comment to explain it, write a loop instead.

## 2 · Iterators and generators

Lists hold everything in memory. A **generator** yields one item at a time, so a 10 GB log file can be processed with a few kilobytes of RAM. This is the single most useful idea in this notebook.

Any object with `__iter__` and `__next__` is an iterator. You rarely write those methods; you write a function with `yield`.

In [ ]:
def countdown(n):
    print("starting")
    while n > 0:
        yield n
        n -= 1
    print("done")

gen = countdown(3)
print(type(gen))          # nothing has run yet — note "starting" is not printed
print(next(gen))
print(next(gen))
print(list(gen))          # drains the rest

Generators are **lazy**: the body only runs when something asks for the next value.
Watch memory: a list of ten million squares vs. a generator of them.

In [ ]:
import sys
squares_list = [i * i for i in range(10_000_000)]
squares_gen  = (i * i for i in range(10_000_000))   # parentheses → generator expression

print(f"list : {sys.getsizeof(squares_list) / 1e6:6.1f} MB")
print(f"gen  : {sys.getsizeof(squares_gen):6d} bytes")
print(sum(squares_gen))   # consumes it; you can only iterate a generator once
del squares_list

### Reading a large file lazily

`data/generated/server.log` has 500 lines; the same code would work on 500 million.
Each stage below is a generator, so the whole pipeline streams line by line.

In [ ]:
from pathlib import Path

# locate the repo's data folder no matter which notebook directory we are in
DATA = next(p / "data" / "generated" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "make_data.py").exists())
LOG = DATA / "server.log"

def read_lines(path):
    with open(path) as f:
        for line in f:
            yield line.rstrip("\n")

def only_errors(lines):
    for line in lines:
        if " ERROR " in line:
            yield line

def latencies(lines):
    for line in lines:
        yield int(line.split("latency_ms=")[1])

pipeline = latencies(only_errors(read_lines(LOG)))
errs = list(pipeline)
print(len(errs), "error lines; mean latency", sum(errs) / len(errs))

### The `itertools` toolbox

The standard library ships lazy building blocks. Three you will actually use:

In [ ]:
from itertools import islice, chain, groupby

first5 = list(islice(read_lines(LOG), 5))          # take the first 5 without reading the file
print(*first5, sep="\n")

print(list(chain([1, 2], (3, 4), "ab")))            # lazily concatenate iterables

# groupby groups *consecutive* equal keys — sort first if the data is not ordered
levels = sorted(line.split()[1] for line in read_lines(LOG))
for level, grp in groupby(levels):
    print(f"{level:8s} {sum(1 for _ in grp)}")

## 3 · `lambda`, `map`, `filter`, `sorted`

A `lambda` is an anonymous one-expression function. Its natural home is as the `key=` argument of
`sorted`, `min`, `max` — and later, Pandas' `.apply()`.

In [ ]:
records = [("Ahmad", 3.4, 2), ("Sara", 3.9, 4), ("Omar", 2.1, 1), ("Lina", 3.9, 3)]

by_gpa = sorted(records, key=lambda r: r[1], reverse=True)
by_gpa_then_year = sorted(records, key=lambda r: (-r[1], r[2]))   # tuple keys for tie-breaks
best = max(records, key=lambda r: r[1])

print(by_gpa)
print(by_gpa_then_year)
print("top:", best[0])

`map` and `filter` return lazy iterators. In modern Python a comprehension is usually clearer,
but you will meet `map`/`filter` constantly in other people's code, so read both fluently.

In [ ]:
names = list(map(lambda r: r[0].upper(), records))
seniors = list(filter(lambda r: r[2] >= 3, records))
print(names)
print(seniors)

# equivalent comprehensions
print([r[0].upper() for r in records] == names)
print([r for r in records if r[2] >= 3] == seniors)

# map with a built-in function and several iterables
print(list(map(pow, [2, 3, 4], [2, 2, 2])))

## 4 · Decorators

A decorator is a function that takes a function and returns a new function.
`@timer` above a `def` is just sugar for `f = timer(f)`.
Use them for cross-cutting concerns: timing, caching, logging, retries.

In [ ]:
import time
from functools import wraps

def timer(func):
    @wraps(func)                      # keeps the original name and docstring
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        result = func(*args, **kwargs)
        dt = time.perf_counter() - t0
        print(f"{func.__name__} took {dt*1000:.1f} ms")
        return result
    return wrapper

@timer
def slow_sum(n):
    """Sum the first n integers the slow way."""
    return sum(i for i in range(n))

slow_sum(2_000_000)
print(slow_sum.__name__, "-", slow_sum.__doc__)

### `functools.lru_cache` — memoisation for free

Expensive pure functions (feature computation, API lookups) can be cached with one line.

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

t0 = time.perf_counter(); fib(80); print(f"first call : {(time.perf_counter()-t0)*1e3:.3f} ms")
t0 = time.perf_counter(); fib(80); print(f"cached call: {(time.perf_counter()-t0)*1e6:.1f} µs")
print(fib.cache_info())

### Decorators with arguments

One more level of nesting: a *factory* that returns a decorator.

In [ ]:
def retry(times):
    def decorator(func):
        @wraps(func)
        def wrapper(*a, **kw):
            for attempt in range(1, times + 1):
                try:
                    return func(*a, **kw)
                except ValueError as e:
                    print(f"attempt {attempt} failed: {e}")
            raise RuntimeError(f"{func.__name__} failed after {times} attempts")
        return wrapper
    return decorator

import random
random.seed(1)

@retry(times=3)
def flaky_download():
    if random.random() < 0.6:
        raise ValueError("network hiccup")
    return "data!"

print(flaky_download())

## 5 · Error handling

Real datasets contain `"N/A"`, empty strings, and numbers with commas. Your code has three choices
for each bad value: crash, skip, or substitute. `try/except` lets you make that choice explicitly.

In [ ]:
raw = ["3.4", "N/A", "", "2.75", "3,1", "4.0"]

def parse_gpa(s):
    try:
        return float(s)
    except ValueError:
        return None          # explicit "missing" marker

cleaned = [parse_gpa(s) for s in raw]
print(cleaned)
print("valid:", [g for g in cleaned if g is not None])

### The full `try / except / else / finally` form

* `except` runs on error, `else` runs only when there was none, `finally` always runs.
* Catch the **narrowest** exception that makes sense. A bare `except:` hides bugs.

In [ ]:
def load_number(path):
    try:
        f = open(path)
    except FileNotFoundError:
        print(f"{path}: not found, using default")
        return 0
    else:
        with f:
            return float(f.read())
    finally:
        print("load_number finished")

print(load_number("does_not_exist.txt"))

### Raising your own exceptions

Define a small exception hierarchy for your project so callers can catch exactly what they care about.

In [ ]:
class DataError(Exception):
    """Base class for problems with input data."""

class SchemaError(DataError):
    pass

def check_row(row, required=("student_id", "gpa")):
    missing = [k for k in required if k not in row]
    if missing:
        raise SchemaError(f"missing columns: {missing}")
    return row

try:
    check_row({"student_id": 1})
except DataError as e:          # catches SchemaError too
    print(type(e).__name__, "->", e)

## 6 · Type hints

Hints do not change how Python runs, but they document intent, let VS Code autocomplete
correctly, and let Pylance warn you *before* you execute. Hover over `summarise` below in VS Code.

In [ ]:
from typing import Iterable, Optional

def mean(values: Iterable[float]) -> float:
    values = list(values)
    return sum(values) / len(values)

def summarise(name: str, gpas: list[float], threshold: Optional[float] = None) -> dict[str, float | int]:
    out = {"n": len(gpas), "mean": mean(gpas)}
    if threshold is not None:
        out["above"] = sum(g > threshold for g in gpas)
    return out

print(summarise("DS", [3.1, 2.8, 3.7], threshold=3.0))

### Dataclasses — typed records in one line each

A `@dataclass` generates `__init__`, `__repr__` and `__eq__` from the annotations.
It is the right tool for small structured records before you reach for Pandas.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Student:
    student_id: int
    name: str
    gpa: float
    courses: list[str] = field(default_factory=list)

    def standing(self) -> str:
        return "honours" if self.gpa >= 3.5 else "good" if self.gpa >= 2.0 else "probation"

s = Student(20230001, "Sara Khalil", 3.8, ["606315"])
print(s)
print(s.standing(), s == Student(20230001, "Sara Khalil", 3.8, ["606315"]))

## 7 · Context managers

`with` guarantees clean-up — the file is closed, the lock released, the timer stopped — even if an
exception is raised in the block. Write your own with `contextlib.contextmanager`.

In [ ]:
from contextlib import contextmanager

@contextmanager
def timed(label):
    t0 = time.perf_counter()
    try:
        yield                       # the with-block runs here
    finally:
        print(f"[{label}] {(time.perf_counter() - t0)*1000:.1f} ms")

with timed("build list"):
    _ = [i * i for i in range(1_000_000)]

with timed("sum generator"):
    _ = sum(i * i for i in range(1_000_000))

In [ ]:
# Several context managers in one `with`
import tempfile, os
with tempfile.TemporaryDirectory() as tmp, open(os.path.join(tmp, "x.txt"), "w") as f:
    f.write("hello")
    print("exists inside:", os.path.exists(f.name))
print("exists after :", os.path.exists(f.name))   # the directory was removed for us

## Putting it together — a tiny ETL in pure Python

Everything from this notebook in one short pipeline over `students.csv`:
generator-based reading, `try/except` parsing, a dataclass, a timing context manager, and a sorted summary.

In [ ]:
import csv

@dataclass
class Row:
    student_id: int
    major: str
    gpa: Optional[float]

def rows(path: Path):
    with open(path, newline="") as f:
        for r in csv.DictReader(f):
            try:
                gpa = float(r["gpa"])
            except ValueError:
                gpa = None
            yield Row(int(r["student_id"]), r["major"].strip(), gpa)

with timed("ETL"):
    data = list(rows(DATA / "students.csv"))

valid = [r for r in data if r.gpa is not None]
by_major: dict[str, list[float]] = {}
for r in valid:
    by_major.setdefault(r.major, []).append(r.gpa)

print(f"{len(data)} rows, {len(data) - len(valid)} unparseable GPAs\n")
for major, g in sorted(by_major.items(), key=lambda kv: -mean(kv[1])):
    print(f"{major:18s} n={len(g):3d}  mean GPA={mean(g):.2f}")

## Summary

* **Comprehensions** replace short accumulate-loops; keep them to one line.
* **Generators** make pipelines lazy; chain them to stream big files.
* **`lambda`** belongs in `key=`; prefer comprehensions over `map`/`filter` in new code.
* **Decorators** add behaviour around functions — `@timer`, `@lru_cache`, `@retry`.
* **Exceptions**: catch narrowly, decide explicitly what a bad value becomes.
* **Type hints + dataclasses** make records self-describing and editor-checkable.
* **Context managers** guarantee clean-up.

Next: open `02_exercises.ipynb`. Every exercise has a check cell — run it to grade yourself.